# Language on the Viability Manifold — companion notebook
**Author:** H. Hamam
**Project:** `Language_Viability_Manifold`
**Outputs:** `MyDrive/Outputs/Language_Viability_Manifold/` (tables, figures, corpora, `outputs_summary.json`)

This notebook (final configuration, Google Colab) implements the prespecified experimental program of the manuscript *Language on the Viability Manifold: A UVIF Framework for Emergent Linguistic Laws*.

**Evidence gate.** QUICK-mode outputs are engineering diagnostics, including adverse results, and are never confirmatory evidence. The full configuration is prespecified and hashed before execution; it should be externally preregistered before any confirmatory run. No threshold may be changed after inspecting full-run outcomes. All thresholds, weights, grids, seeds and step counts live in the `CONFIG` cell and are written to `config.json` before any experiment runs. Every number that reaches the manuscript must be traceable to a CSV in `tables/` produced by this notebook.

| Block | Manuscript section | Hypotheses / results tested |
|---|---|---|
| A. Multilingual corpus mapping | Exp. A | law observables only (no meanings ⇒ no viability) |
| B. Emergent communication vs baselines | Exp. B | H1, H2, Thm. insufficiency (scrambling), Thm. feasibility (frontier) |
| C. Ablation, force-ratio sweep, Jacobian, hysteresis | Exp. C | H3, H6, H7, hysteresis diagnostic |
| D. Iterated transmission | Exp. D | H4, H5 (contraction), Q_S |
| H8. Context-cover certificate | §Context tests | H8 |

Set `QUICK = True` for a smoke test (minutes on CPU). Set `QUICK = False` for the preregistered run (GPU recommended).

In [ ]:
# ---------------- 0. Environment, Drive, output directory ----------------
import os, sys, json, platform, subprocess, datetime, math
from pathlib import Path
QUICK = False           # False = preregistered full configuration (3000 steps, 5 seeds, 256 meanings). True = smoke test only.
# MODE selects which cells run under "Run all"; skipped cells print a one-line notice instead of stopping the run.
#   "decoding_diagnostic" : cell 6c only (train/eval decoding-temperature check; not registered)
#   "pilot_B"             : Experiments A and B only
#   "capacity_pilot"      : A, B, then the capacity pilot (6b); not registered
#   "optimizer_pilot"     : cell 6d only (optimiser-hygiene settings compared on task_only / uvif_full; not registered)
#   "full"                : the complete preregistered grid (A, B, C, D, H8, figures, summary)
MODE = "full"
_RUN = {"decoding_diagnostic": {"decoding"}, "optimizer_pilot": {"optimizer"}, "pilot_B": {"A", "B"}, "capacity_pilot": {"A", "B", "capacity"},
        "full": {"A", "B", "C", "D", "H8", "figures", "summary"}}[MODE]
def stage(name):
    on = name in _RUN
    if not on: print(f"[MODE={MODE}] stage '{name}' skipped.")
    return on
PROJECT_NAME = "Language_Viability_Manifold"
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    OUT_DIR = f"/content/drive/MyDrive/Outputs/{PROJECT_NAME}"
except Exception as e:
    print("Google Drive not mounted (", type(e).__name__, ") -- falling back to a local directory")
    OUT_DIR = os.path.abspath(f"./Outputs/{PROJECT_NAME}")
for sub in ("tables", "figures", "corpora"): os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
try:
    import conllu  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "conllu"], check=False)
print("Output directory:", OUT_DIR)

In [ ]:
# ---------------- 1. CONFIG (preregistered; written before any experiment) ----------------
CONFIG = dict(
    author="H. Hamam", project=PROJECT_NAME, created=datetime.datetime.now().isoformat(), quick=QUICK,
    # meaning space and channel
    n_attr=3 if QUICK else 4, n_val=4, prior_skew=0.8, A=8, max_len=6 if QUICK else 8, hidden=64,   # full run: 256 meanings
    eta_train=0.10, eta_eval_grid=[0.0, 0.05, 0.10, 0.20, 0.30],
    # training
    # warmup_frac: non-epistemic forces ramp linearly over the first 30% of steps, identically in every condition
    lr=3e-3, batch=128, gumbel_tau=1.0, warmup_frac=0.3,
    # optimiser hygiene (condition-blind; defaults reproduce the v7 behaviour): cosine lr decay to 0, gradient-norm clip, Gumbel tau annealed linearly from gumbel_tau to gumbel_tau_final
    lr_cosine=True, grad_clip=1.0, gumbel_tau_final=None,   # registered: cosine decay + clipping, identical in every condition
    steps=300 if QUICK else 3000, learn_steps=60 if QUICK else 400, eval_T=2048 if QUICK else 8192,
    seeds=[0, 1] if QUICK else [0, 1, 2, 3, 4],
    conditions=["uvif_full", "generic_sum", "least_effort", "task_only", "random"],
    # viability thresholds (a priori) and ULVI weights (equal = primary analysis)
    tau=dict(E=0.60, P=0.60, C=0.30, F=0.30, S=0.50), w=dict(E=1.0, P=1.0, C=1.0, F=1.0, S=1.0),
    tau_train=dict(E=0.60, P=0.60, C=0.30, F=0.30, S=0.50), hinge_weight=10.0,
    # loss normalisation scales a_i (fixed a priori; enter lambda_eff)
    # a_i = 1/scale_i: E,P = per-attribute cross-entropy at chance (ln n_val); C = listener mean |w| budget;
    # F = normalised length; S = max per-position policy entropy (ln(A+1)).  Fixed before any run.
    loss_scales=dict(E=math.log(4), P=2*math.log(4), C=0.20, F=1.00, S=math.log(9)),
    # Q_C = 1 - mean|w|/complexity_budget (primary); cutoff-based sparsity is a reported sensitivity only
    complexity_budget=0.20, active_parameter_cutoff=1e-3, ulvi_floor=1e-12,
    # Experiment C
    lambda_grid=[0.25, 0.5, 1.0, 2.0, 4.0] if QUICK else [0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0], n_boot=500,
    jacobian_step=0.10, jacobian_seeds=2 if QUICK else 5, rank_tol=1e-2,
    hysteresis_grid=[0.5, 1.0, 2.0, 4.0] if QUICK else [0.25, 0.5, 1.0, 2.0, 4.0, 8.0], hysteresis_steps=100 if QUICK else 800, hysteresis_seeds=1 if QUICK else 3,
    # Experiment D
    generations=4 if QUICK else 12, bottleneck=0.6, imitation_steps=100 if QUICK else 600, interaction_steps=100 if QUICK else 800,
    # H8 context cover (contexts = evaluation channel noise)
    context_cover=[0.0, 0.1, 0.2, 0.3], lipschitz_inflation=2.0,
    # Experiment A
    languages=["en", "fr", "de", "es", "ar", "zh", "tr", "fi"] if not QUICK else ["en", "fr"],
    ud_note="Universal Dependencies r2.18 test splits; URL and SHA-256 are recorded in expA_corpus_laws.csv",
)
with open(os.path.join(OUT_DIR, "config.json"), "w") as f: json.dump(CONFIG, f, indent=2)
import hashlib
CONFIG_SHA256 = hashlib.sha256(json.dumps({k: v for k, v in CONFIG.items() if k != "created"}, sort_keys=True).encode()).hexdigest()
print("CONFIG sha256 (excluding timestamp):", CONFIG_SHA256)
env = dict(python=platform.python_version(), platform=platform.platform())
try:
    import torch, numpy, scipy, pandas; env.update(torch=torch.__version__, numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, cuda=torch.cuda.is_available())
except Exception as e: env["import_error"] = str(e)
with open(os.path.join(OUT_DIR, "environment.json"), "w") as f: json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

## 2. Core library — estimators, viability, scrambling operator

In [ ]:
# ---------------------------------------------------------------------------
# UVIF Linguistic Viability — core library
# Author metadata: H. Hamam
# This cell defines every estimator used in the notebook. Nothing here is
# tuned to any outcome; thresholds and weights are read from CONFIG only.
# ---------------------------------------------------------------------------
import math, json, os, time, itertools, random, collections
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import stats, optimize
from pathlib import Path
_trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy < 2.0 compatibility

# ------------------------------ utilities ----------------------------------
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def save_df(df, name):
    p = os.path.join(OUT_DIR, "tables", name); df.to_csv(p, index=False); return p

def save_json(obj, name):
    p = os.path.join(OUT_DIR, name)
    with open(p, "w") as f: json.dump(obj, f, indent=2, default=float)
    return p

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT_DIR, "figures", f"{name}.{ext}"), dpi=200, bbox_inches="tight")

# ------------------------------ information ---------------------------------
def entropy_bits(counts):
    c = np.asarray(counts, dtype=float); c = c[c > 0]; p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def mutual_information_bits(x, y, bias_correct=True):
    """Plug-in MI with Miller–Madow correction (bits)."""
    x = np.asarray(x); y = np.asarray(y); T = len(x)
    ct = pd.crosstab(x, y).values.astype(float)
    hx = entropy_bits(ct.sum(1)); hy = entropy_bits(ct.sum(0)); hxy = entropy_bits(ct.ravel())
    mi = hx + hy - hxy
    if bias_correct:
        # Miller--Madow: H_MM(X)+H_MM(Y)-H_MM(X,Y).
        # The earlier notebook used the opposite sign.
        kx = (ct.sum(1) > 0).sum(); ky = (ct.sum(0) > 0).sum(); kxy = (ct > 0).sum()
        mi += ((kx - 1) + (ky - 1) - (kxy - 1)) / (2 * T * math.log(2))
    return float(max(mi, 0.0))

def binary_entropy(e):
    e = min(max(e, 1e-12), 1 - 1e-12); return float(-e * math.log2(e) - (1 - e) * math.log2(1 - e))

def symmetric_channel_capacity(eta, A):
    """Per-symbol capacity (bits) of the A-ary symmetric substitution channel."""
    if eta <= 0: return math.log2(A)
    return float(math.log2(A) - binary_entropy(eta) - eta * math.log2(A - 1))

def fano_min_length(H_M, eps, n_meanings, C):
    """Eq. (converse): minimal expected length compatible with error eps."""
    num = H_M - binary_entropy(eps) - eps * math.log2(max(n_meanings - 1, 1))
    return float(max(num, 0.0) / max(C, 1e-12))

# ------------------------------ linguistic laws -----------------------------
def zipf_fit(freqs):
    """Rank–frequency exponent.  Primary: discrete power-law MLE on frequencies
    (Clauset et al. style, xmin=1).  Secondary: OLS on log rank / log freq.
    Also returns a KS distance of the empirical rank curve from the fitted
    power law (spectrum-only diagnostic)."""
    f = np.sort(np.asarray(freqs, dtype=float))[::-1]; f = f[f > 0]
    if len(f) < 5: return dict(alpha_ols=np.nan, alpha_mle=np.nan, ks=np.nan, n_types=len(f))
    r = np.arange(1, len(f) + 1)
    slope, intercept = np.polyfit(np.log(r), np.log(f), 1)
    alpha_ols = -slope
    # MLE for the discrete Zipf (zeta) distribution over ranks, via 1-D search
    def nll(a):
        z = np.sum(r ** (-a)); return a * np.sum(f * np.log(r)) + f.sum() * np.log(z)
    grid = np.linspace(0.2, 3.0, 57); a0 = grid[np.argmin([nll(a) for a in grid])]
    res = optimize.minimize_scalar(nll, bounds=(max(0.05, a0 - 0.3), a0 + 0.3), method="bounded")
    alpha_mle = float(res.x)
    p_emp = f / f.sum(); p_fit = r ** (-alpha_mle); p_fit /= p_fit.sum()
    ks = float(np.max(np.abs(np.cumsum(p_emp) - np.cumsum(p_fit))))
    return dict(alpha_ols=float(alpha_ols), alpha_mle=alpha_mle, ks=ks, n_types=int(len(f)))

def heaps_fit(tokens, n_points=30):
    tokens = list(tokens); N = len(tokens)
    if N < 50: return dict(beta=np.nan)
    seen = set(); V = []; idx = np.unique(np.geomspace(10, N, n_points).astype(int))
    j = 0; V_vals = []
    for i, t in enumerate(tokens, 1):
        seen.add(t)
        if j < len(idx) and i == idx[j]: V_vals.append(len(seen)); j += 1
    idx = idx[:len(V_vals)]
    beta, _ = np.polyfit(np.log(idx), np.log(V_vals), 1)
    return dict(beta=float(beta))

def abbreviation_fit(type_freq, type_len):
    """Zipf's law of abbreviation: association between frequency and length."""
    f = np.asarray(type_freq, float); l = np.asarray(type_len, float)
    if len(f) < 5 or np.std(l) == 0: return dict(rho_spearman=np.nan, tau_kendall=np.nan)
    return dict(rho_spearman=float(stats.spearmanr(f, l).correlation),
                tau_kendall=float(stats.kendalltau(f, l).correlation))

def menzerath_fit(construct_sizes, mean_constituent_sizes):
    """Menzerath–Altmann: log(mean constituent) ~ b*log(construct size)."""
    x = np.asarray(construct_sizes, float); y = np.asarray(mean_constituent_sizes, float)
    m = (x > 0) & (y > 0)
    if m.sum() < 5: return dict(b=np.nan)
    b, _ = np.polyfit(np.log(x[m]), np.log(y[m]), 1); return dict(b=float(b))

def spectrum_statistics(signals):
    """All spectrum-only statistics of a list of signals (tuples of ints)."""
    counter = collections.Counter(signals)
    types = list(counter.keys()); freqs = np.array([counter[t] for t in types]); lens = np.array([len(t) for t in types])
    out = {}
    out.update({f"zipf_{k}": v for k, v in zipf_fit(freqs).items()})
    out.update({f"heaps_{k}": v for k, v in heaps_fit(signals).items()})
    out.update({f"abbrev_{k}": v for k, v in abbreviation_fit(freqs, lens).items()})
    out["mean_length"] = float(np.mean([len(s) for s in signals]))
    out["signal_entropy_bits"] = entropy_bits(freqs)
    return out

# ------------------------------ viability -----------------------------------
FORCES = ["E", "P", "C", "F", "S"]

def ulvi(Q, w=None, floor=1e-12):
    """Eq. (ulvi).  Exact-zero behaviour is intended; the floor only prevents
    log underflow and is declared in CONFIG."""
    w = w or {k: 1.0 for k in FORCES}; W = sum(w.values())
    logs = sum(w[k] * math.log(max(Q[k], floor)) for k in FORCES) / W
    return float(math.exp(logs)) if all(Q[k] > 0 for k in FORCES) else 0.0

def viable(Q, tau): return all(Q[k] >= tau[k] for k in FORCES)

def viability_margin(Q, tau, L):
    """Certified margin mu = min_i (Q_i - tau_i)/L_i   (Theorem margin)."""
    return float(min((Q[k] - tau[k]) / max(L[k], 1e-9) for k in FORCES))

# ------------------------------ scrambling ----------------------------------
def semantic_scramble(meanings, signals, rng):
    """Definition (scramble): permute meaning labels, keep signals verbatim."""
    perm = rng.permutation(len(meanings)); return [meanings[i] for i in perm], list(signals)


## 3. Agents, objectives, evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Emergent-communication agents, training objectives, and quality estimators
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MeaningSpace:
    """Factorial meaning space with a Zipf-skewed prior (context parameter)."""
    def __init__(self, n_attr, n_val, prior_skew=0.0, seed=0):
        self.n_attr, self.n_val = n_attr, n_val
        self.meanings = np.array(list(itertools.product(range(n_val), repeat=n_attr)))
        self.n = len(self.meanings)
        rng = np.random.default_rng(seed); order = rng.permutation(self.n)
        ranks = np.empty(self.n); ranks[order] = np.arange(1, self.n + 1)
        p = ranks ** (-prior_skew); self.p = p / p.sum()
        self.H = entropy_bits(self.p)
        onehots = np.zeros((self.n, n_attr * n_val), dtype=np.float32)
        for i, m in enumerate(self.meanings):
            for a, v in enumerate(m): onehots[i, a * n_val + v] = 1.0
        self.X = torch.tensor(onehots, device=DEVICE)
        self.Y = torch.tensor(self.meanings, device=DEVICE, dtype=torch.long)
    def sample(self, B, rng):
        return torch.tensor(rng.choice(self.n, size=B, p=self.p), device=DEVICE)

class Speaker(nn.Module):
    def __init__(self, in_dim, A, max_len, hidden):
        super().__init__(); self.A, self.max_len = A, max_len
        self.enc = nn.Linear(in_dim, hidden); self.gru = nn.GRUCell(A + 1, hidden)
        self.out = nn.Linear(hidden, A + 1)   # index 0 = EOS
    def forward(self, x, temperature=1.0, hard=True):
        """Returns one-hot symbols (B,T,A+1) via straight-through Gumbel-softmax,
        per-position log-probs, and a soft 'alive' mask (1 before EOS)."""
        B = x.shape[0]; h = torch.tanh(self.enc(x)); inp = torch.zeros(B, self.A + 1, device=x.device)
        syms, logps, alive_list = [], [], []; alive = torch.ones(B, device=x.device)
        for t in range(self.max_len):
            h = self.gru(inp, h); logits = self.out(h)
            if t == 0: logits = logits - 1e4 * F.one_hot(torch.zeros(B, dtype=torch.long, device=x.device), self.A + 1)  # the empty string is not a signal
            logp = F.log_softmax(logits, -1)
            y = F.gumbel_softmax(logits, tau=temperature, hard=hard)
            syms.append(y); logps.append(logp); alive_list.append(alive)
            alive = alive * (1.0 - y[:, 0]); inp = y
        return torch.stack(syms, 1), torch.stack(logps, 1), torch.stack(alive_list, 1)

class Listener(nn.Module):
    def __init__(self, A, n_attr, n_val, hidden):
        super().__init__(); self.emb = nn.Linear(A + 1, hidden, bias=False)
        self.gru = nn.GRU(hidden, hidden, batch_first=True); self.heads = nn.Linear(hidden, n_attr * n_val)
        self.n_attr, self.n_val = n_attr, n_val
    def forward(self, syms, alive):
        e = self.emb(syms) * alive.unsqueeze(-1); _, h = self.gru(e)
        return self.heads(h[-1]).view(-1, self.n_attr, self.n_val)

def channel(syms, alive, eta, rng_t):
    """A-ary symmetric substitution channel applied to alive, non-EOS symbols."""
    if eta <= 0: return syms
    B, T, K = syms.shape
    flip = (torch.rand(B, T, device=syms.device, generator=rng_t) < eta).float() * alive * (1 - syms[:, :, 0])
    rand_idx = torch.randint(1, K, (B, T), device=syms.device, generator=rng_t)
    rand_oh = F.one_hot(rand_idx, K).float()
    return syms * (1 - flip.unsqueeze(-1)) + rand_oh * flip.unsqueeze(-1)

def decode_symbols(syms, alive):
    idx = syms.argmax(-1).cpu().numpy(); al = (alive > 0.5).cpu().numpy(); out = []
    for i in range(idx.shape[0]):
        s = tuple(int(v) for v, a in zip(idx[i], al[i]) if a and v != 0); out.append(s if s else (0,))
    return out

def listener_loss(logits, y):
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))

def accuracy(logits, y): return (logits.argmax(-1) == y).all(-1).float().mean().item()

# ------------------------------ objectives ----------------------------------
def force_losses(spk, lst, ms, B, rng, eta_train, cfg, gen):
    """Returns the five raw force losses (differentiable) and clean CE."""
    m = ms.sample(B, rng); x = ms.X[m]; y = ms.Y[m]
    syms, logps, alive = spk(x, temperature=cfg["gumbel_tau"])
    ce_clean = listener_loss(lst(syms, alive), y)
    ce_noisy = listener_loss(lst(channel(syms, alive, eta_train, gen), alive), y)
    L = {}
    L["E"] = ce_clean
    # Protection is performance under corruption plus an explicit degradation
    # penalty.  Keeping ce_clean attached makes this distinct from merely adding
    # a second noisy epistemic loss.
    L["P"] = ce_noisy + F.relu(ce_noisy - ce_clean)
    # Differentiable parsimony surrogate; evaluation reports effective sparsity.
    L["C"] = sum(p.abs().mean() for p in lst.parameters()) / len(list(lst.parameters()))
    L["F"] = alive.sum(1).mean() / spk.max_len
    ent = -(logps.exp() * logps).sum(-1) * alive
    # Declared transmission surrogate. Its correlation with measured fresh-learner
    # performance is an empirical construct-validity check, not an identity.
    L["S"] = ent.sum(1).mean() / spk.max_len
    return L, ce_clean

def objective(L, condition, w, cfg, ramp=1.0):
    """Training objective per condition.  Normalised surrogate qualities
    Qt_i = 1 - L_i/scale_i are used only for the constraint hinge and the
    geometric term; evaluation qualities are computed separately.
    `ramp` in [0,1] is the preregistered warm-up multiplier applied identically
    to every non-epistemic force in every condition (curriculum, not tuning)."""
    sc = cfg["loss_scales"]; w = {k: (w[k] if k == "E" else w[k] * ramp) for k in FORCES}
    if condition == "task_only":     return L["E"]
    if condition == "least_effort":  return L["E"] + w["F"] * L["F"]
    if condition == "generic_sum":   return sum(w[k] * L[k] / sc[k] for k in FORCES)
    if condition == "uvif_full":
        Qt = {k: torch.clamp(1 - L[k] / sc[k], 1e-4, 1.0) for k in FORCES}
        hinge = sum(F.relu(cfg["tau_train"][k] - Qt[k]) ** 2 * (1.0 if k == "E" else ramp) for k in FORCES) * cfg["hinge_weight"]
        geo = -sum(w[k] * torch.log(Qt[k]) for k in FORCES) / max(sum(w.values()), 1e-9)
        return hinge + geo
    raise ValueError(condition)

def train_pair(condition, w, cfg, seed, ms, steps=None, eta_train=None, init=None, log_every=0):
    set_seed(seed); rng = np.random.default_rng(seed); gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    spk = Speaker(ms.X.shape[1], cfg["A"], cfg["max_len"], cfg["hidden"]).to(DEVICE)
    lst = Listener(cfg["A"], ms.n_attr, ms.n_val, cfg["hidden"]).to(DEVICE)
    if init is not None: spk.load_state_dict(init[0]); lst.load_state_dict(init[1])
    opt = torch.optim.Adam(list(spk.parameters()) + list(lst.parameters()), lr=cfg["lr"])
    steps = steps or cfg["steps"]; eta_train = cfg["eta_train"] if eta_train is None else eta_train
    if condition == "random":  # untrained speaker, trained listener only
        opt = torch.optim.Adam(lst.parameters(), lr=cfg["lr"])
    hist = []
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps) if cfg.get("lr_cosine") else None
    tau0, tau1 = cfg["gumbel_tau"], cfg.get("gumbel_tau_final") or cfg["gumbel_tau"]
    for it in range(steps):
        cfg_step = dict(cfg); cfg_step["gumbel_tau"] = tau0 + (tau1 - tau0) * it / max(1, steps - 1)
        L, ce = force_losses(spk, lst, ms, cfg["batch"], rng, eta_train, cfg_step, gen)
        ramp = min(1.0, it / max(1, cfg["warmup_frac"] * steps)) if init is None else 1.0
        loss = L["E"] if condition == "random" else objective(L, condition, w, cfg, ramp)
        opt.zero_grad(); loss.backward()
        if cfg.get("grad_clip"): torch.nn.utils.clip_grad_norm_(list(spk.parameters()) + list(lst.parameters()), cfg["grad_clip"])
        opt.step()
        if sched is not None: sched.step()
        if log_every and it % log_every == 0: hist.append(dict(step=it, loss=loss.item(), ce=ce.item()))
    return spk, lst, hist

# ------------------------------ evaluation ----------------------------------
@torch.no_grad()
def emit(spk, ms, m_idx):
    syms, _, alive = spk(ms.X[m_idx], temperature=1e-3, hard=True); return syms, alive

@torch.no_grad()
def evaluate_code(spk, lst, ms, cfg, seed, eta_grid=None, learnability=True):
    """Five qualities, spectrum statistics, and the (eps, L, C) triple."""
    rng = np.random.default_rng(seed + 1000); gen = torch.Generator(device=DEVICE); gen.manual_seed(seed + 1000)
    m = ms.sample(cfg["eval_T"], rng); y = ms.Y[m]; syms, alive = emit(spk, ms, m)
    logits = lst(syms, alive); acc0 = accuracy(logits, y)
    m_hat = (logits.argmax(-1) * (ms.n_val ** torch.arange(ms.n_attr, device=DEVICE))).sum(-1).cpu().numpy()
    m_np = m.cpu().numpy(); I_hat = mutual_information_bits(m_np, m_hat)
    Q = {}
    Q["E"] = float(np.clip(I_hat / ms.H, 0, 1))
    eta_grid = eta_grid or cfg["eta_eval_grid"]; curve = []
    for eta in eta_grid:
        curve.append(accuracy(lst(channel(syms, alive, eta, gen), alive), y))
    # Absolute noisy semantic performance: a useless code can no longer score
    # as perfectly protected merely because noise does not make it worse.
    noisy_information = []
    for eta in eta_grid:
        noisy = channel(syms, alive, eta, gen)
        noisy_logits = lst(noisy, alive)
        noisy_hat = (noisy_logits.argmax(-1) * (ms.n_val ** torch.arange(ms.n_attr, device=DEVICE))).sum(-1).cpu().numpy()
        noisy_information.append(mutual_information_bits(m_np, noisy_hat) / ms.H)
    Q["P"] = float(np.clip(np.mean(noisy_information), 0, 1))
    # Effective sparsity is still a model-parsimony proxy, not a claim about human
    # cognitive cost. Sensitivity to the cutoff must be reported.
    # Primary: L1 budget parsimony (same quantity as the training surrogate L_C).
    pars = torch.cat([p.detach().abs().reshape(-1) for p in lst.parameters()])
    mean_abs_w = float(pars.mean())
    Q["C"] = float(np.clip(1 - mean_abs_w / cfg["complexity_budget"], 0, 1))
    # Sensitivity only: cutoff-based sparsity (v6 primary; degenerate at hidden=64, kept for reporting).
    active_fraction = float((pars > cfg["active_parameter_cutoff"]).float().mean())
    sparsity_cutoff = float(np.clip(1 - active_fraction, 0, 1))
    n_params = int(pars.numel())
    signals = decode_symbols(syms, alive); mean_len = float(np.mean([len(s) for s in signals]))
    Q["F"] = float(np.clip(1 - (mean_len - 1) / (cfg["max_len"] - 1), 0, 1))
    if learnability:
        Q["S"] = learnability_quality(spk, ms, cfg, seed)
    else: Q["S"] = float("nan")
    spec = spectrum_statistics(signals)
    eps = 1 - acc0
    out = dict(Q=Q, acc_clean=acc0, eps=eps, mean_len=mean_len, robustness_curve=curve,
               robustness_retention=float(np.mean(curve) / max(acc0, 1e-6)), signals=signals,
               meanings=m_np.tolist(), spectrum=spec, mean_abs_w=mean_abs_w, sparsity_cutoff=sparsity_cutoff, n_params=n_params,
               C_eta_train=symmetric_channel_capacity(cfg["eta_train"], cfg["A"]),
               fano_L_min=fano_min_length(ms.H, eps, ms.n, symmetric_channel_capacity(cfg["eta_train"], cfg["A"])))
    return out

def learnability_quality(spk, ms, cfg, seed):
    """Q_S surrogate for a single generation: a fresh listener trained for a
    fixed budget on the frozen code; reported as normalised information."""
    set_seed(seed + 7); rng = np.random.default_rng(seed + 7)
    lst = Listener(cfg["A"], ms.n_attr, ms.n_val, cfg["hidden"]).to(DEVICE); opt = torch.optim.Adam(lst.parameters(), lr=cfg["lr"])
    with torch.enable_grad():
        for _ in range(cfg["learn_steps"]):
            m = ms.sample(cfg["batch"], rng); syms, alive = emit(spk, ms, m)
            loss = listener_loss(lst(syms, alive), ms.Y[m]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        m = ms.sample(cfg["eval_T"], rng); syms, alive = emit(spk, ms, m); logits = lst(syms, alive)
        m_hat = (logits.argmax(-1) * (ms.n_val ** torch.arange(ms.n_attr, device=DEVICE))).sum(-1).cpu().numpy()
        return float(np.clip(mutual_information_bits(m.cpu().numpy(), m_hat) / ms.H, 0, 1))

@torch.no_grad()
def code_descriptor(spk, ms):
    """Descriptor: empirical P(signal-type | meaning) as a dict; metric d = mean TV."""
    syms, alive = emit(spk, ms, torch.arange(ms.n, device=DEVICE)); sig = decode_symbols(syms, alive)
    return {i: s for i, s in enumerate(sig)}

def descriptor_distance(d1, d2):
    return float(np.mean([d1[i] != d2[i] for i in d1]))   # Hamming over meanings (TV for deterministic codes)

# ------------------------------ scrambling test -----------------------------
def _signals_to_tensor(signals, cfg):
    """Encode decoded signals back to one-hot (B,T,A+1) plus alive mask."""
    B, T, K = len(signals), cfg["max_len"], cfg["A"] + 1
    idx = torch.zeros(B, T, dtype=torch.long); alive = torch.zeros(B, T)
    for i, s in enumerate(signals):
        for t, v in enumerate(s[:T]): idx[i, t] = v; alive[i, t] = 1.0
    return F.one_hot(idx, K).float().to(DEVICE), alive.to(DEVICE)

def recoverable_information(meanings, signals, ms, cfg, seed):
    """Q_E of a (meaning, signal) record: a fresh listener is trained for the
    learnability budget on a split of the record and evaluated on the rest.
    This is the estimator the scrambling test must use; plug-in MI over signal
    types saturates at H(M) whenever most signals are unique (v6 defect)."""
    set_seed(seed + 11); rng = np.random.default_rng(seed + 11)
    m = np.asarray(meanings); n = len(m); perm = rng.permutation(n); tr, te = perm[: n // 2], perm[n // 2 :]
    X, alive = _signals_to_tensor(signals, cfg); Y = ms.Y[torch.tensor(m, device=DEVICE)]
    lst = Listener(cfg["A"], ms.n_attr, ms.n_val, cfg["hidden"]).to(DEVICE); opt = torch.optim.Adam(lst.parameters(), lr=cfg["lr"])
    with torch.enable_grad():
        for _ in range(cfg["learn_steps"]):
            b = torch.tensor(rng.choice(tr, cfg["batch"]), device=DEVICE)
            loss = listener_loss(lst(X[b], alive[b]), Y[b]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        b = torch.tensor(te, device=DEVICE); logits = lst(X[b], alive[b])
        m_hat = (logits.argmax(-1) * (ms.n_val ** torch.arange(ms.n_attr, device=DEVICE))).sum(-1).cpu().numpy()
        return float(np.clip(mutual_information_bits(m[te], m_hat) / ms.H, 0, 1))

def scramble_test(res, ms, cfg, seed):
    """Empirical check of Theorem (insufficiency): spectrum invariant, Q_E -> 0.
    Q_E before/after is measured by retraining a fresh listener on the record
    (recoverable_information).  The plug-in signal-type MI is reported only as
    a diagnostic of the estimator floor."""
    rng = np.random.default_rng(seed + 99)
    m_s, s_s = semantic_scramble(res["meanings"], res["signals"], rng)
    spec_s = spectrum_statistics(s_s)
    QE_o = recoverable_information(res["meanings"], res["signals"], ms, cfg, seed)
    QE_s = recoverable_information(m_s, s_s, ms, cfg, seed)
    sig2idx = {s: i for i, s in enumerate(set(s_s))}
    plug_o = mutual_information_bits(res["meanings"], [sig2idx[s] for s in res["signals"]]) / ms.H
    plug_s = mutual_information_bits(m_s, [sig2idx[s] for s in s_s]) / ms.H
    return dict(spectrum_original=res["spectrum"], spectrum_scrambled=spec_s,
                QE_original=QE_o, QE_scrambled=QE_s,
                plugin_QE_original=float(np.clip(plug_o, 0, 1)), plugin_QE_scrambled=float(np.clip(plug_s, 0, 1)),
                n_signal_types=len(sig2idx), n_samples=len(s_s))


## 4. Experiment drivers

In [ ]:
# ===================== Experiment A: multilingual corpus mapping =====================
UD_SOURCES = {   # Universal Dependencies treebanks (test splits), GitHub raw; release pinned in CONFIG
    "en": "UD_English-EWT/r2.18/en_ewt-ud-test.conllu",
    "fr": "UD_French-GSD/r2.18/fr_gsd-ud-test.conllu",
    "de": "UD_German-GSD/r2.18/de_gsd-ud-test.conllu",
    "es": "UD_Spanish-GSD/r2.18/es_gsd-ud-test.conllu",
    "ar": "UD_Arabic-PADT/r2.18/ar_padt-ud-test.conllu",
    "zh": "UD_Chinese-GSD/r2.18/zh_gsd-ud-test.conllu",
    "tr": "UD_Turkish-IMST/r2.18/tr_imst-ud-test.conllu",
    "fi": "UD_Finnish-TDT/r2.18/fi_tdt-ud-test.conllu",
}

def fetch_conllu(lang):
    import urllib.request, hashlib
    url = "https://raw.githubusercontent.com/UniversalDependencies/" + UD_SOURCES[lang]
    path = os.path.join(OUT_DIR, "corpora", f"{lang}.conllu")
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    return path, url, digest

def corpus_laws(path):
    from conllu import parse_incr
    tokens, sent_len, sent_mean_wlen = [], [], []
    with open(path, encoding="utf-8") as f:
        for sent in parse_incr(f):
            words = [t["form"].lower() for t in sent if isinstance(t["id"], int) and t["upos"] != "PUNCT"]
            if not words: continue
            tokens += words; sent_len.append(len(words)); sent_mean_wlen.append(np.mean([len(w) for w in words]))
    counter = collections.Counter(tokens); types = list(counter); freqs = np.array([counter[t] for t in types])
    lens = np.array([len(t) for t in types])
    out = dict(n_tokens=len(tokens), n_types=len(types), entropy_bits=entropy_bits(freqs))
    out.update({f"zipf_{k}": v for k, v in zipf_fit(freqs).items()})
    out.update({f"heaps_{k}": v for k, v in heaps_fit(tokens).items()})
    out.update({f"abbrev_{k}": v for k, v in abbreviation_fit(freqs, lens).items()})
    # Menzerath at the sentence/word level: construct = sentence (words), constituent = word (characters)
    df = pd.DataFrame(dict(L=sent_len, w=sent_mean_wlen)).groupby("L")["w"].mean().reset_index()
    out.update({f"menzerath_{k}": v for k, v in menzerath_fit(df["L"], df["w"]).items()})
    out["mean_word_len_chars"] = float(np.mean([len(t) for t in tokens]))
    return out

def run_experiment_A(cfg):
    rows = []
    for lang in cfg["languages"]:
        try:
            path, url, digest = fetch_conllu(lang); r = corpus_laws(path); r.update(lang=lang, source=url, sha256=digest, status="ok")
        except Exception as e:
            r = dict(lang=lang, status=f"unavailable: {type(e).__name__}: {e}")
        rows.append(r); print(lang, r.get("status"), {k: round(v, 3) for k, v in r.items() if isinstance(v, float)})
    df = pd.DataFrame(rows); save_df(df, "expA_corpus_laws.csv"); return df

# ===================== Experiment B: emergent communication vs baselines ==============
def run_condition(condition, w, cfg, seed, ms, **kw):
    spk, lst, _ = train_pair(condition, w, cfg, seed, ms, **kw)
    res = evaluate_code(spk, lst, ms, cfg, seed)
    Q = res["Q"]; row = dict(condition=condition, seed=seed, **{f"Q_{k}": Q[k] for k in FORCES})
    row.update(ULVI=ulvi(Q, cfg["w"]), viable=viable(Q, cfg["tau"]), acc_clean=res["acc_clean"], eps=res["eps"],
               mean_len=res["mean_len"], fano_L_min=res["fano_L_min"], C_eta=res["C_eta_train"],
               frontier_slack=res["mean_len"] - res["fano_L_min"],
               mean_abs_w=res["mean_abs_w"], sparsity_cutoff=res["sparsity_cutoff"], n_params=res["n_params"])
    row.update({k: v for k, v in res["spectrum"].items()})
    return row, (spk, lst, res)

def run_experiment_B(cfg, ms):
    rows, models = [], {}
    for cond in cfg["conditions"]:
        for seed in cfg["seeds"]:
            row, mod = run_condition(cond, cfg["w"], cfg, seed, ms); rows.append(row); models[(cond, seed)] = mod
            qs = ", ".join("%s:%.2f" % (k, row["Q_" + k]) for k in FORCES)
            print(f"[B] {cond:14s} seed={seed} ULVI={row['ULVI']:.3f} viable={row['viable']} Q={{{qs}}} alpha={row['zipf_alpha_mle']:.2f} len={row['mean_len']:.2f}")
    df = pd.DataFrame(rows); save_df(df, "expB_conditions.csv")
    # ---- Theorem (insufficiency): scrambling test on every trained code ----
    sc_rows = []
    for (cond, seed), (spk, lst, res) in models.items():
        t = scramble_test(res, ms, cfg, seed)
        sc_rows.append(dict(condition=cond, seed=seed, QE_original=t["QE_original"], QE_scrambled=t["QE_scrambled"],
                            plugin_QE_original=t["plugin_QE_original"], plugin_QE_scrambled=t["plugin_QE_scrambled"],
                            n_signal_types=t["n_signal_types"], n_samples=t["n_samples"],
                            alpha_original=t["spectrum_original"]["zipf_alpha_mle"], alpha_scrambled=t["spectrum_scrambled"]["zipf_alpha_mle"],
                            beta_original=t["spectrum_original"]["heaps_beta"], beta_scrambled=t["spectrum_scrambled"]["heaps_beta"],
                            abbrev_original=t["spectrum_original"]["abbrev_rho_spearman"], abbrev_scrambled=t["spectrum_scrambled"]["abbrev_rho_spearman"]))
    dfs = pd.DataFrame(sc_rows); save_df(dfs, "expB_scrambling_test.csv")
    return df, dfs, models

# ===================== Experiment C: ablation, force-ratio sweep, Jacobian, hysteresis ====
def run_ablations(cfg, ms):
    rows = []
    for k in FORCES:
        w = dict(cfg["w"]); w[k] = 0.0
        for seed in cfg["seeds"]:
            row, _ = run_condition("uvif_full", w, cfg, seed, ms); row["ablated"] = k; rows.append(row)
            print(f"[C-ablate] -{k} seed={seed} ULVI={row['ULVI']:.3f} len={row['mean_len']:.2f} alpha={row['zipf_alpha_mle']:.2f}")
    df = pd.DataFrame(rows); save_df(df, "expC_ablations.csv"); return df

def run_force_ratio_sweep(cfg, ms):
    """H6: log alpha vs log lambda_eff.  lambda_eff = (a_F w_F)/(a_E w_E) with a_i = 1/scale_i."""
    rows = []
    for ratio in cfg["lambda_grid"]:
        w = dict(cfg["w"]); w["F"] = ratio * w["E"]
        lam_eff = (w["F"] / cfg["loss_scales"]["F"]) / (w["E"] / cfg["loss_scales"]["E"])
        for seed in cfg["seeds"]:
            row, _ = run_condition("uvif_full", w, cfg, seed, ms); row.update(ratio=ratio, lambda_eff=lam_eff); rows.append(row)
            print(f"[C-sweep] ratio={ratio:.3f} seed={seed} alpha={row['zipf_alpha_mle']:.2f} len={row['mean_len']:.2f}")
    df = pd.DataFrame(rows); save_df(df, "expC_force_ratio_sweep.csv")
    d = df.dropna(subset=["zipf_alpha_mle"]); d = d[d["zipf_alpha_mle"] > 0]
    fit = None
    if len(d) >= 4 and d["lambda_eff"].nunique() >= 3:
        X = np.log(d["lambda_eff"]); Y = np.log(d["zipf_alpha_mle"]); res = stats.linregress(X, Y)
        boot = []
        rng = np.random.default_rng(0)
        for _ in range(cfg["n_boot"]):
            i = rng.integers(0, len(d), len(d))
            if np.unique(X.values[i]).size < 2: continue
            boot.append(stats.linregress(X.values[i], Y.values[i]).slope)
        fit = dict(slope=res.slope, intercept=res.intercept, r2=res.rvalue ** 2, slope_ci95=[float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                   predicted_slope=1.0, predicted_intercept=-math.log(math.log(cfg["A"])))
    return df, fit

def run_jacobian(cfg, ms, base_w=None):
    """H7: finite-difference intervention-response Jacobian around the base point."""
    base_w = base_w or cfg["w"]; h = cfg["jacobian_step"]; ys = {}
    def response(w, seed):
        row, _ = run_condition("uvif_full", w, cfg, seed, ms)
        return np.array([row[f"Q_{k}"] for k in FORCES] + [row["zipf_alpha_mle"], row["heaps_beta"], row["mean_len"]])
    J_list = []
    for seed in cfg["seeds"][: cfg["jacobian_seeds"]]:
        cols = []
        for k in FORCES:
            wp = dict(base_w); wm = dict(base_w); wp[k] = base_w[k] * (1 + h); wm[k] = base_w[k] * (1 - h)
            yp = response(wp, seed); ym = response(wm, seed)
            dlog = math.log(1 + h) - math.log(1 - h)
            cols.append((yp - ym) / dlog)                # exact log-weight denominator
        J = np.stack(cols, 1); J = np.nan_to_num(J); J_list.append(J)
        print(f"[C-jacobian] seed={seed} singular values={np.round(np.linalg.svd(J, compute_uv=False), 3)}")
    J_mean = np.mean(J_list, 0); sv = np.linalg.svd(J_mean, compute_uv=False)
    out = dict(J_mean=J_mean.tolist(), singular_values=sv.tolist(), rank=int(np.linalg.matrix_rank(J_mean, tol=cfg["rank_tol"])),
               condition_number=float(sv[0] / max(sv[-1], 1e-12)), sigma_min=float(sv[-1]),
               response_names=[f"Q_{k}" for k in FORCES] + ["alpha", "beta", "mean_len"])
    save_json(out, "expC_jacobian.json"); return out

def run_hysteresis(cfg, ms):
    """Exploratory: sweep w_F up then down with warm starts; loop area of mean length."""
    grid = list(cfg["hysteresis_grid"]); rows = []
    for seed in cfg["seeds"][: cfg["hysteresis_seeds"]]:
        init = None
        for direction, g in (("up", grid), ("down", grid[::-1])):
            for wf in g:
                w = dict(cfg["w"]); w["F"] = wf
                spk, lst, _ = train_pair("uvif_full", w, cfg, seed, ms, steps=cfg["hysteresis_steps"], init=init)
                init = (spk.state_dict(), lst.state_dict()); res = evaluate_code(spk, lst, ms, cfg, seed, learnability=False)
                rows.append(dict(seed=seed, direction=direction, w_F=wf, mean_len=res["mean_len"], Q_E=res["Q"]["E"]))
        d = pd.DataFrame([r for r in rows if r["seed"] == seed])
        up = d[d.direction == "up"].sort_values("w_F"); dn = d[d.direction == "down"].sort_values("w_F")
        area = float(_trapz(up["mean_len"].values - dn["mean_len"].values, up["w_F"].values))
        print(f"[C-hysteresis] seed={seed} loop area (mean length) = {area:.4f}")
        rows.append(dict(seed=seed, direction="loop_area", w_F=np.nan, mean_len=area, Q_E=np.nan))
    df = pd.DataFrame(rows); save_df(df, "expC_hysteresis.csv"); return df

# ===================== Experiment D: iterated transmission ===========================
def run_iterated_transmission(cfg, ms, condition="uvif_full"):
    """Generation g+1 learns from a bottlenecked sample of generation g's signals
    (supervised imitation), then interacts under the condition's objective."""
    rows = []
    for seed in cfg["seeds"]:
        set_seed(seed); rng = np.random.default_rng(seed)
        spk, lst, _ = train_pair(condition, cfg["w"], cfg, seed, ms); prev_desc = code_descriptor(spk, ms); dists = []; survived = None
        for g in range(cfg["generations"]):
            # --- bottleneck: the learner sees only a fraction of meanings ---
            n_seen = int(cfg["bottleneck"] * ms.n); seen = torch.tensor(rng.choice(ms.n, n_seen, replace=False), device=DEVICE)
            syms_t, alive_t = emit(spk, ms, seen)
            new_spk = Speaker(ms.X.shape[1], cfg["A"], cfg["max_len"], cfg["hidden"]).to(DEVICE); opt = torch.optim.Adam(new_spk.parameters(), lr=cfg["lr"])
            target = syms_t.argmax(-1)
            for _ in range(cfg["imitation_steps"]):
                _, logps, _ = new_spk(ms.X[seen], temperature=cfg["gumbel_tau"])
                loss = -(logps.gather(-1, target.unsqueeze(-1)).squeeze(-1) * alive_t).sum(1).mean(); opt.zero_grad(); loss.backward(); opt.step()
            spk, lst, _ = train_pair(condition, cfg["w"], cfg, seed + g + 1, ms, steps=cfg["interaction_steps"], init=(new_spk.state_dict(), lst.state_dict()))
            desc = code_descriptor(spk, ms); d = descriptor_distance(desc, prev_desc); dists.append(d); prev_desc = desc
            res = evaluate_code(spk, lst, ms, cfg, seed, learnability=False); Q = dict(res["Q"]); Q["S"] = float(np.clip(1 - d, 0, 1))
            v = viable(Q, cfg["tau"])
            if survived is None and not v: survived = g
            rows.append(dict(condition=condition, seed=seed, generation=g, drift=d, viable=v, ULVI=ulvi(Q, cfg["w"]),
                             **{f"Q_{k}": Q[k] for k in FORCES}, acc_clean=res["acc_clean"], mean_len=res["mean_len"], alpha=res["spectrum"]["zipf_alpha_mle"]))
            print(f"[D] {condition} seed={seed} g={g} drift={d:.3f} acc={res['acc_clean']:.2f} viable={v}")
        # contraction modulus: geometric regression of successive drifts (rho = exp(slope))
        dd = np.array(dists); dd = dd[dd > 0]
        rho = float(np.exp(stats.linregress(np.arange(len(dd)), np.log(dd)).slope)) if len(dd) >= 3 else np.nan
        rows.append(dict(condition=condition, seed=seed, generation=-1, drift=np.nan, viable=np.nan, ULVI=np.nan, rho_estimate=rho,
                         first_nonviable_generation=survived if survived is not None else cfg["generations"]))
    df = pd.DataFrame(rows); save_df(df, f"expD_transmission_{condition}.csv"); return df

# ===================== Context-cover certificate (H8) ================================
def run_context_cover(cfg, ms, models):
    """Empirical H8 transfer diagnostic over channel noise.

    Adjacent finite differences are not certified global Lipschitz bounds. We
    therefore label these as candidate certificates and report sensitivity to a
    preregistered inflation factor; theorem-level certification requires an
    analytic or independently validated upper bound.
    """
    cover = list(cfg["context_cover"]); eps_cover = max(np.diff(cover)) / 2; held = [(a + b) / 2 for a, b in zip(cover[:-1], cover[1:])]
    rows = []
    for (cond, seed), (spk, lst, _) in models.items():
        Qz = {z: evaluate_code(spk, lst, ms, cfg, seed, eta_grid=[z], learnability=False)["Q"] for z in cover + held}
        # conservative Lipschitz estimate in z from adjacent cover points
        Lz = {k: max(abs(Qz[a][k] - Qz[b][k]) / (b - a) for a, b in zip(cover[:-1], cover[1:])) for k in ["E", "P", "C", "F"]}
        Lz_safe = {k: cfg["lipschitz_inflation"] * v for k, v in Lz.items()}
        cert = all(Qz[z][k] >= cfg["tau"][k] + Lz_safe[k] * eps_cover for z in cover for k in Lz)
        held_ok = all(all(Qz[z][k] >= cfg["tau"][k] for k in Lz) for z in held)
        cover_ok = all(all(Qz[z][k] >= cfg["tau"][k] for k in Lz) for z in cover)
        rows.append(dict(condition=cond, seed=seed, certified=cert, viable_on_cover=cover_ok, viable_on_heldout=held_ok,
                         false_certificate=bool(cert and not held_ok), certificate_status="empirical_candidate",
                         **{f"Lz_{k}": v for k, v in Lz.items()}))
    df = pd.DataFrame(rows); save_df(df, "expH8_context_cover.csv"); return df


## 5. Run — Experiment A (corpus law observables)

In [ ]:
cfg = CONFIG
if stage("A"):
    t0 = time.time()
    dfA = run_experiment_A(cfg); display(dfA.drop(columns=["source"], errors="ignore"))
    print(f"Experiment A: {time.time()-t0:.0f}s")

## 6. Run — Experiment B (baselines, scrambling test, admissibility frontier)

In [ ]:
if stage("B"):
    t0 = time.time(); ms = MeaningSpace(cfg["n_attr"], cfg["n_val"], cfg["prior_skew"], seed=0); print("meanings:", ms.n, "H(M) bits:", round(ms.H, 3))
    dfB, dfScr, models = run_experiment_B(cfg, ms)
    summaryB = dfB.groupby("condition")[[f"Q_{k}" for k in FORCES] + ["ULVI", "viable", "acc_clean", "mean_len", "zipf_alpha_mle", "heaps_beta", "abbrev_rho_spearman", "frontier_slack"]].mean()
    display(summaryB.round(3)); save_df(summaryB.reset_index(), "expB_summary_by_condition.csv")
    print("\nScrambling test (Theorem insufficiency): spectrum must be identical, listener-recoverable Q_E must collapse; plugin_* columns show the signal-type MI floor")
    print("Q_C sensitivity (cutoff-based sparsity vs L1-budget primary):"); print(dfB[["condition", "seed", "Q_C", "sparsity_cutoff", "mean_abs_w", "n_params"]].round(3).to_string())
    display(dfScr.round(3))
    print("Frontier check (Theorem feasibility): any negative frontier_slack is a calibration/assumption violation to report:")
    print(dfB[["condition", "seed", "mean_len", "fano_L_min", "frontier_slack"]].round(3).to_string())
    print(f"Experiment B: {time.time()-t0:.0f}s")


## 6b. Capacity pilot (diagnostic, not registered)

In [ ]:
if stage("capacity"):
    # ---------------- Capacity pilot (NOT part of the registered analysis) ----------------
    # Question: is the failure to reach tau_E = 0.60 at (hidden=64, steps=3000) an optimisation/capacity budget
    # limit or a property of the objective?  task_only is the pure-epistemic reference: if it cannot reach tau_E,
    # no objective can, and the thresholds bind by construction.  Thresholds and all other constants are unchanged.
    t0 = time.time(); cap_rows = []
    CAP_GRID = [dict(hidden=64, steps=10000), dict(hidden=128, steps=6000), dict(hidden=128, steps=10000)]
    CAP_SEEDS = cfg["seeds"][:3]; CAP_CONDITIONS = ["task_only", "uvif_full"]
    for g in CAP_GRID:
        cfg_g = dict(cfg); cfg_g.update(g)
        for cond in CAP_CONDITIONS:
            for seed in CAP_SEEDS:
                row, _ = run_condition(cond, cfg_g["w"], cfg_g, seed, ms); row.update(hidden=g["hidden"], steps=g["steps"]); cap_rows.append(row)
                qs = ", ".join("%s:%.2f" % (k, row["Q_" + k]) for k in FORCES)
                print(f"[cap] hidden={g['hidden']} steps={g['steps']} {cond:10s} seed={seed} viable={row['viable']} Q={{{qs}}} len={row['mean_len']:.2f}  ({time.time()-t0:.0f}s)")
    # reference rows from the registered budget already in dfB
    ref = dfB[dfB.condition.isin(CAP_CONDITIONS)].copy(); ref["hidden"] = cfg["hidden"]; ref["steps"] = cfg["steps"]
    dfCap = pd.concat([ref, pd.DataFrame(cap_rows)], ignore_index=True); save_df(dfCap, "pilot_capacity.csv")
    capsum = dfCap.groupby(["hidden", "steps", "condition"])[[f"Q_{k}" for k in FORCES] + ["ULVI", "viable", "mean_len"]].mean().round(3)
    display(capsum)
    save_json(dict(project=PROJECT_NAME, author="H. Hamam", purpose="capacity pilot; not registered", finished=datetime.datetime.now().isoformat(),
                   tau=cfg["tau"], grid=CAP_GRID, seeds=CAP_SEEDS, summary=capsum.reset_index().to_dict("records")), "pilot_capacity_summary.json")
    print(f"Capacity pilot: {time.time()-t0:.0f}s")


## 6c. Decoding-temperature diagnostic (not registered)

In [ ]:
if stage("decoding"):
    # ---------------- Decoding-temperature diagnostic (NOT part of the registered analysis) ----------------
    # Hypothesis: training samples signals at gumbel_tau = 1.0 while evaluation decodes greedily (temperature 1e-3).
    # Conditions without entropy pressure (task_only) may keep a high-entropy policy, so the greedy string differs
    # from anything the listener was trained on.  Test: evaluate Q_E under greedy vs. training-temperature sampling,
    # and report speaker policy entropy and the task_only training curve.
    t0 = time.time()
    ms = MeaningSpace(cfg["n_attr"], cfg["n_val"], cfg["prior_skew"], seed=0)
    DEC_TEMPS = [1e-3, 0.5, 1.0]; DEC_SEEDS = cfg["seeds"][:2]; DEC_CONDS = ["task_only", "uvif_full"]

    @torch.no_grad()
    def qe_at_temperature(spk, lst, ms, cfg, seed, temperature, n_rep=1):
        rng = np.random.default_rng(seed + 1000); m = ms.sample(cfg["eval_T"], rng); y = ms.Y[m]
        accs, qes, ents = [], [], []
        for r in range(n_rep):
            torch.manual_seed(seed + 5000 + r)
            syms, logps, alive = spk(ms.X[m], temperature=temperature, hard=True)
            logits = lst(syms, alive); accs.append(accuracy(logits, y))
            m_hat = (logits.argmax(-1) * (ms.n_val ** torch.arange(ms.n_attr, device=DEVICE))).sum(-1).cpu().numpy()
            qes.append(mutual_information_bits(m.cpu().numpy(), m_hat) / ms.H)
            ent = (-(logps.exp() * logps).sum(-1) * alive).sum(1) / alive.sum(1).clamp(min=1)
            ents.append(float(ent.mean()) / math.log(2))
        return dict(acc=float(np.mean(accs)), Q_E=float(np.clip(np.mean(qes), 0, 1)), policy_entropy_bits_per_pos=float(np.mean(ents)))

    rows, curves = [], []
    for cond in DEC_CONDS:
        for seed in DEC_SEEDS:
            spk, lst, hist = train_pair(cond, cfg["w"], cfg, seed, ms, log_every=100)
            for h in hist: curves.append(dict(condition=cond, seed=seed, **h))
            for T in DEC_TEMPS:
                r = qe_at_temperature(spk, lst, ms, cfg, seed, T, n_rep=3 if T > 0.01 else 1)
                r.update(condition=cond, seed=seed, decode_temperature=T); rows.append(r)
                print(f"[dec] {cond:10s} seed={seed} T={T:<6} acc={r['acc']:.3f} Q_E={r['Q_E']:.3f} H_policy={r['policy_entropy_bits_per_pos']:.2f} bits/pos  ({time.time()-t0:.0f}s)")
    dfDec = pd.DataFrame(rows); save_df(dfDec, "pilot_decoding_temperature.csv")
    dfCurve = pd.DataFrame(curves); save_df(dfCurve, "pilot_training_curves.csv")
    display(dfDec.pivot_table(index=["condition", "seed"], columns="decode_temperature", values=["Q_E", "acc"]).round(3))
    print("\nTraining cross-entropy (clean listener CE, nats) every 100 steps:")
    display(dfCurve.pivot_table(index="step", columns=["condition", "seed"], values="ce").round(3).iloc[::5])
    save_json(dict(project=PROJECT_NAME, author="H. Hamam", purpose="decoding-temperature diagnostic; not registered", finished=datetime.datetime.now().isoformat(),
                   rows=rows), "pilot_decoding_summary.json")
    print(f"Decoding diagnostic: {time.time()-t0:.0f}s")


## 6d. Optimiser-hygiene pilot (not registered)

In [ ]:
# ---------------- Optimiser-hygiene pilot (NOT part of the registered analysis) ----------------
# The decoding diagnostic showed near-deterministic policies and noisy, non-converging training CE.  Three
# condition-blind optimiser settings are compared on task_only and uvif_full; thresholds, weights, scales,
# architecture and budget are unchanged.  The chosen setting is applied identically to every condition.
if stage("optimizer"):
    t0 = time.time()
    ms = MeaningSpace(cfg["n_attr"], cfg["n_val"], cfg["prior_skew"], seed=0)
    OPT_SETTINGS = {
        "a_current":         dict(),
        "b_cosine_clip":     dict(lr=1e-3, lr_cosine=True, grad_clip=1.0),
        "c_cosine_clip_tau": dict(lr=1e-3, lr_cosine=True, grad_clip=1.0, gumbel_tau_final=0.3),
        "d_cosine_clip_lr3":  dict(lr=3e-3, lr_cosine=True, grad_clip=1.0),
    }
    OPT_SEEDS = cfg["seeds"][:2]; OPT_CONDS = ["task_only", "uvif_full"]
    rows, curves = [], []
    for name, over in OPT_SETTINGS.items():
        cfg_o = dict(cfg); cfg_o.update(over)
        for cond in OPT_CONDS:
            for seed in OPT_SEEDS:
                spk, lst, hist = train_pair(cond, cfg_o["w"], cfg_o, seed, ms, log_every=100)
                for h in hist: curves.append(dict(setting=name, condition=cond, seed=seed, **h))
                res = evaluate_code(spk, lst, ms, cfg_o, seed); Q = res["Q"]
                row = dict(setting=name, condition=cond, seed=seed, **{f"Q_{k}": Q[k] for k in FORCES}, ULVI=ulvi(Q, cfg["w"]), viable=viable(Q, cfg["tau"]),
                           acc_clean=res["acc_clean"], mean_len=res["mean_len"], ce_final=hist[-1]["ce"], ce_min=min(h["ce"] for h in hist),
                           ce_last5_std=float(np.std([h["ce"] for h in hist[-5:]])))
                rows.append(row)
                qs = ", ".join("%s:%.2f" % (k, Q[k]) for k in FORCES)
                print(f"[opt] {name:18s} {cond:10s} seed={seed} viable={row['viable']} Q={{{qs}}} acc={res['acc_clean']:.2f} len={res['mean_len']:.2f} ce_final={row['ce_final']:.3f}  ({time.time()-t0:.0f}s)")
    dfOpt = pd.DataFrame(rows); save_df(dfOpt, "pilot_optimizer.csv")
    dfOptCurve = pd.DataFrame(curves); save_df(dfOptCurve, "pilot_optimizer_curves.csv")
    display(dfOpt.groupby(["setting", "condition"])[["Q_E", "Q_P", "Q_C", "Q_F", "Q_S", "ULVI", "viable", "acc_clean", "mean_len", "ce_final", "ce_last5_std"]].mean().round(3))
    print("\nTraining CE (nats) every 300 steps:")
    display(dfOptCurve.pivot_table(index="step", columns=["setting", "condition"], values="ce").round(3).iloc[::3])
    save_json(dict(project=PROJECT_NAME, author="H. Hamam", purpose="optimiser-hygiene pilot; not registered", finished=datetime.datetime.now().isoformat(),
                   settings=OPT_SETTINGS, seeds=OPT_SEEDS, rows=rows), "pilot_optimizer_summary.json")
    print(f"Optimiser pilot: {time.time()-t0:.0f}s")


## 7. Run — Experiment C (ablations, H6 force-ratio slope, H7 Jacobian, hysteresis)

In [ ]:
if stage("C"):
    t0 = time.time()
    dfAbl = run_ablations(cfg, ms); display(dfAbl.groupby("ablated")[[f"Q_{k}" for k in FORCES] + ["ULVI", "mean_len", "zipf_alpha_mle"]].mean().round(3))
    dfSweep, fitH6 = run_force_ratio_sweep(cfg, ms); print("H6 fit:", json.dumps(fitH6, indent=2) if fitH6 else "insufficient valid alpha estimates")
    jac = run_jacobian(cfg, ms); print("H7 Jacobian: rank =", jac["rank"], " sigma_min =", round(jac["sigma_min"], 4), " cond =", round(jac["condition_number"], 2))
    dfHys = run_hysteresis(cfg, ms)
    print(f"Experiment C: {time.time()-t0:.0f}s")

## 8. Run — Experiment D (iterated transmission, contraction modulus, H5)

In [ ]:
if stage("D"):
    t0 = time.time()
    dfD = pd.concat([run_iterated_transmission(cfg, ms, c) for c in ["uvif_full", "least_effort"]], ignore_index=True)
    gen_rows = dfD[dfD.generation >= 0]; meta = dfD[dfD.generation < 0]
    display(gen_rows.groupby(["condition", "generation"])[["drift", "acc_clean", "ULVI", "viable"]].mean().round(3))
    display(meta[["condition", "seed", "rho_estimate", "first_nonviable_generation"]])
    print(f"Experiment D: {time.time()-t0:.0f}s")

## 9. Run — H8 context-cover certificate

In [ ]:
if stage("H8"):
    dfH8 = run_context_cover(cfg, ms, models); display(dfH8.round(3))
    print("false-certificate rate:", dfH8["false_certificate"].mean() if len(dfH8) else "n/a")

## 10. Figures

In [ ]:
if stage("figures"):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    # Fig 1 — analytic admissibility frontier (Theorem feasibility): no data needed.
    # Matches the manuscript caption: |M|=256 equiprobable meanings (H=8 bits), A=8, L_max=8.
    N_FIG, H_FIG, A_FIG, LMAX_FIG = 256, 8.0, cfg["A"], 8
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.4)); eps = np.linspace(0.001, 0.6, 200)
    for eta in [0.0, 0.05, 0.1, 0.2, 0.3]:
        C = symmetric_channel_capacity(eta, A_FIG); Lmin = np.array([fano_min_length(H_FIG, e, N_FIG, C) for e in eps])
        ax1.plot(eps, Lmin, label=f"η={eta}, C={C:.2f} b/sym")
        # panel (b): largest admissible efficiency threshold tau_F = 1 - L_min/L_max for epistemic threshold tau_E = 1 - eps
        ax2.plot(1 - eps, np.clip(1 - Lmin / LMAX_FIG, 0, 1), label=f"η={eta}")
    ax1.set_xlabel("semantic error ε"); ax1.set_ylabel("minimal expected length L"); ax1.set_title("(a) Admissibility frontier (Fano converse)"); ax1.legend(fontsize=7)
    ax2.set_xlabel("epistemic threshold τ_E = 1 − ε"); ax2.set_ylabel("largest admissible τ_F"); ax2.set_title("(b) Threshold frontier"); ax2.legend(fontsize=7)
    save_fig(fig, "fig_frontier")
    # Fig 2 — Zipf fit vs ULVI map (H2)
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for cond, d in dfB.groupby("condition"): ax.scatter(1 - d["zipf_ks"], d["ULVI"], label=cond, s=30)
    ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("Zipf conformity (1 − KS)"); ax.set_ylabel("ULVI"); ax.legend(fontsize=7); ax.set_title("Zipfianity vs viability"); save_fig(fig, "fig_zipf_vs_ulvi")
    # Fig 3 — scrambling test
    fig, ax = plt.subplots(figsize=(5, 3.4)); x = np.arange(len(dfScr))
    ax.bar(x - 0.2, dfScr["QE_original"], 0.4, label="Q_E original"); ax.bar(x + 0.2, dfScr["QE_scrambled"], 0.4, label="Q_E scrambled")
    ax.set_xticks(x); ax.set_xticklabels([f"{c}\n{s}" for c, s in zip(dfScr.condition, dfScr.seed)], fontsize=6); ax.legend(fontsize=7); ax.set_title("Semantic scrambling: Q_E collapses, α unchanged"); save_fig(fig, "fig_scrambling")
    # Fig 4 — H6 slope
    if fitH6:
        fig, ax = plt.subplots(figsize=(5, 3.4)); d = dfSweep[dfSweep.zipf_alpha_mle > 0]
        ax.scatter(np.log(d.lambda_eff), np.log(d.zipf_alpha_mle), s=25); xx = np.linspace(np.log(d.lambda_eff).min(), np.log(d.lambda_eff).max(), 10)
        ax.plot(xx, fitH6["intercept"] + fitH6["slope"] * xx, label=f"fit slope={fitH6['slope']:.2f}"); ax.plot(xx, fitH6["predicted_intercept"] + xx, "--", label="prediction slope=1")
        ax.set_xlabel("log λ_eff"); ax.set_ylabel("log α"); ax.legend(fontsize=7); ax.set_title("H6: exponent vs calibrated force ratio"); save_fig(fig, "fig_H6_slope")
    # Fig 5 — transmission
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for cond, d in gen_rows.groupby("condition"): g = d.groupby("generation")["ULVI"].mean(); ax.plot(g.index, g.values, marker="o", label=cond)
    ax.set_xlabel("generation"); ax.set_ylabel("ULVI"); ax.legend(fontsize=7); ax.set_title("Iterated transmission"); save_fig(fig, "fig_transmission")
    print("figures saved to", os.path.join(OUT_DIR, "figures"))

## 11. Outputs summary (single source of truth for the manuscript)

In [ ]:
if stage("summary"):
    summary = dict(
        project=PROJECT_NAME, author="H. Hamam", quick_mode=QUICK, finished=datetime.datetime.now().isoformat(), out_dir=OUT_DIR,
        meaning_space=dict(n=ms.n, H_bits=ms.H), config_file="config.json", environment_file="environment.json",
        expA=dict(languages_ok=dfA[dfA.status == "ok"]["lang"].tolist() if "status" in dfA else [], table="tables/expA_corpus_laws.csv"),
        expB=dict(table="tables/expB_conditions.csv", summary="tables/expB_summary_by_condition.csv",
                  viable_fraction_by_condition=dfB.groupby("condition")["viable"].mean().to_dict(),
                  H2_high_zipf_nonviable=int(((1 - dfB["zipf_ks"] > 0.8) & (~dfB["viable"])).sum()),
                  scrambling=dict(table="tables/expB_scrambling_test.csv", mean_QE_original=float(dfScr.QE_original.mean()), mean_QE_scrambled=float(dfScr.QE_scrambled.mean()),
                                  max_abs_alpha_change=float((dfScr.alpha_original - dfScr.alpha_scrambled).abs().max())),
                  frontier_violations=int((dfB["frontier_slack"] < 0).sum())),
        expC=dict(ablations="tables/expC_ablations.csv", sweep="tables/expC_force_ratio_sweep.csv", H6_fit=fitH6, jacobian="expC_jacobian.json",
                  H7=dict(rank=jac["rank"], sigma_min=jac["sigma_min"], condition_number=jac["condition_number"]),
                  hysteresis_loop_area=dfHys[dfHys.direction == "loop_area"]["mean_len"].tolist()),
        expD=dict(table="tables/expD_transmission_*.csv", rho_estimates=meta[["condition", "seed", "rho_estimate", "first_nonviable_generation"]].to_dict("records")),
        H8=dict(table="tables/expH8_context_cover.csv", false_certificate_rate=float(dfH8["false_certificate"].mean()) if len(dfH8) else None),
        reminder="QUICK=True results are smoke tests and must not enter the manuscript.")
    save_json(summary, "outputs_summary.json"); print(json.dumps(summary, indent=2, default=str))